# Stimulus inspection and report exports

Select the experiment below. Exp 10 uses `stimuli/definitions/`; older generated packages may keep trajectories directly under `stimuli/`.
Inspect time traces, angular coverage, actual spatial radii/dot footprints, and explicit phase timing. Exp 10 standalone controls have no blank padding: bout alone contains only the bout; preconditioning alone ends with its final flicker pulse.
The generator saves the full initial report. This notebook displays the same CSV-derived views; optional saving preserves existing previews first.


In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd

import matplotlib.pyplot as plt
from IPython.display import display

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'src' / 'stimulus_visualization.py').is_file())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.stimulus_visualization import (
    load_stimulus_trajectories, summarize_stimuli,
    plot_stimulus_traces, plot_stimulus_positions,
    plot_stimulus_phase_timing, plot_stimulus_geometry,
)
from src.data_loading import resolve_lab_root


In [ ]:
EXPERIMENT = 'Exp_10_RF_model_distance'
DERIVED_ROOT = resolve_lab_root() / 'Results/Paper/2p_derived'
STIMULI_ROOT = DERIVED_ROOT / EXPERIMENT / 'stimuli'
INPUT_DIR = STIMULI_ROOT / 'definitions' if (STIMULI_ROOT / 'definitions').is_dir() else STIMULI_ROOT
PATTERN = '*_trajectory.csv'
SAVE_PREVIEWS = False  # Generator saves the initial package; enable explicitly to refresh previews.

settings = json.loads((INPUT_DIR / 'parameters/resolved_settings.json').read_text(encoding='utf-8'))
side_settings = [settings[side] for side in ('left', 'right') if side in settings]
rates = {s['framerate'] for s in side_settings}
rotations = {s['rotation_angle_deg'] for s in side_settings}
if len(rates) != 1 or len(rotations) != 1:
    raise ValueError('Inspect hemifields separately when their frame rates or rotations differ.')


In [ ]:
samples = load_stimulus_trajectories(
    INPUT_DIR, pattern=PATTERN, framerate=next(iter(rates)),
    rotation_angle_deg=next(iter(rotations)),
)
summary = summarize_stimuli(samples)
timings = pd.read_csv(INPUT_DIR / 'parameters/stimulus_time_summary.csv')


## Standard representation
Solid lines show angle; dashed lines show the CSV dot size, including off frames. Colors identify dots. Size is not distance from the fish.

In [ ]:
standard_figure = plot_stimulus_traces(samples, cols=3, panel_size=(6, 3.8))
display(standard_figure)
plt.close(standard_figure)

## Spatial coverage

The polar panels show unique visible directions with equal-size markers; their ring does not encode distance. The additional geometry panels show actual radii and displayed dot footprints in mm, with preconditioning and test phases overlaid. The centre arrow follows the configured coordinate rotation.


In [ ]:
polar_figure = plot_stimulus_positions(summary, cols=4, panel_size=(3.8, 3.8))
display(polar_figure)
plt.close(polar_figure)

geometry_figure = plot_stimulus_geometry(samples, timings, cols=4, panel_size=(3.8, 4.0), rotation_angle_deg=next(iter(rotations)))
display(geometry_figure)
plt.close(geometry_figure)


## Explicit phase timing

Coloured bars show initial blank, preconditioning, the blank pause, actual test bouts, and blank observation windows. Black pulses indicate actual dot visibility. External before/after pauses are excluded. An omitted bout is not labelled as motion.


In [ ]:
timing_figure = plot_stimulus_phase_timing(samples, timings, figsize=(12, 8))
display(timing_figure)
plt.close(timing_figure)


## Summary and optional export

The generator writes the initial CSV summary and all four PNGs. SAVE_PREVIEWS defaults to False. Enable it to refresh the standard, polar, geometry, and timing figures after preserving dated copies. For Exp 10, operational details and provenance are in definitions/STIMULUS_DESCRIPTIONS.md and definitions/parameters/.


In [ ]:
display(summary)
if SAVE_PREVIEWS:
    from datetime import datetime
    import shutil
    previous = [INPUT_DIR / name for name in ('stimuli_standard.png', 'stimuli_polar.png',
                                             'stimuli_geometry.png', 'stimuli_timing.png')]
    existing = [path for path in previous if path.exists()]
    if existing:
        backup = STIMULI_ROOT.parent / 'archive' / (datetime.now().strftime('%Y-%m-%d_%H%M%S_%f') + '_previews')
        backup.mkdir(parents=True)
        for path in existing:
            shutil.copy2(path, backup / path.name)
    for name, figure in [('stimuli_standard.png', standard_figure), ('stimuli_polar.png', polar_figure),
                         ('stimuli_geometry.png', geometry_figure), ('stimuli_timing.png', timing_figure)]:
        figure.savefig(INPUT_DIR / name, dpi=300)
    print(f'Saved previews to {INPUT_DIR}')
